In [1]:
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import time
import os
import glob

In [14]:
url = "https://clinicaltrials.gov/api/v2/studies"

terms = [
    "breast prosthesis",
    "breast implant"
]

dfs = []

session = requests.Session()

for term in terms:

    params = {
        "query.term": term,
        "pageSize": 1000
    }

    while True:

        response = session.get(url, params=params)
        response.raise_for_status()

        data = response.json()

        studies = data.get("studies", [])

        if not studies:
            break

        df = pd.json_normalize(studies)
        dfs.append(df)

        next_page = data.get("nextPageToken")

        if not next_page:
            break

        params["pageToken"] = next_page


df_final = pd.concat(dfs, ignore_index=True)

# remove estudos duplicados
df_final = df_final.drop_duplicates(
    subset="protocolSection.identificationModule.nctId"
)

print("Total de estudos:", len(df_final))

Total de estudos: 858


In [15]:
df_final

,hasResults,protocolSection.identificationModule.nctId,protocolSection.identificationModule.orgStudyIdInfo.id,protocolSection.identificationModule.secondaryIdInfos,protocolSection.identificationModule.organization.fullName,protocolSection.identificationModule.organization.class,protocolSection.identificationModule.briefTitle,protocolSection.identificationModule.officialTitle,protocolSection.statusModule.statusVerifiedDate,protocolSection.statusModule.overallStatus,...,protocolSection.statusModule.dispFirstPostDateStruct.type,protocolSection.designModule.nPtrsToThisExpAccNctId,protocolSection.ipdSharingStatementModule.url,protocolSection.identificationModule.nctIdAliases,derivedSection.miscInfoModule.submissionTracking.firstMcpInfo.postDateStruct.date,derivedSection.miscInfoModule.submissionTracking.firstMcpInfo.postDateStruct.type,protocolSection.referencesModule.availIpds,documentSection.largeDocumentModule.noSap,protocolSection.identificationModule.orgStudyIdInfo.type,protocolSection.identificationModule.orgStudyIdInfo.link
0,False,NCT01027637,HFHS-PS1209,"[{'id': 'PS1209', 'type': 'OTHER_GRANT', 'doma...",Henry Ford Health System,OTHER,Regenerative Tissue Matrix for Breast Reconstr...,In Situ Elasticity of Alloderm® in Breast Reco...,2009-12,COMPLETED,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,False,NCT02724371,MEN-15-001,NaN,"Mentor Worldwide, LLC",INDUSTRY,A Study of the Safety and Effectiveness of the...,A Study of the Safety and Effectiveness of the...,2026-03,ACTIVE_NOT_RECRUITING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,False,NCT03981263,RC31/17/0335,NaN,"University Hospital, Toulouse",OTHER,Quality of Life in Patients With Standard Exte...,Study of the Quality of Life in Patients With ...,2025-02,COMPLETED,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,False,NCT01191164,BCCA-H09-01053,NaN,British Columbia Cancer Agency,OTHER,Pilot Study of Partial Breast Irradiation Util...,A Prospective Study to Evaluate the Feasibilit...,2011-08,UNKNOWN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,False,NCT01793168,03-10-014,"[{'id': 'Hypersomnia Foundation', 'type': 'REG...",Sanford Health,OTHER,Rare Disease Patient Registry & Natural Histor...,Coordination of Rare Diseases at Sanford,2025-05,RECRUITING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1074,False,NCT04608175,001,"[{'id': '2019/59-FIS-HUSC', 'type': 'OTHER', '...",University Ramon Llull,OTHER,The Efficacy of Acupuncture as a Complementary...,The Efficacy of Acupuncture as a Complementary...,2020-09,UNKNOWN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1076,False,NCT03734770,PROPER-1,NaN,Universita di Verona,OTHER,Patient's Preferences About Subcutaneous or Va...,Patient's Preferences About Subcutaneous or Va...,2022-05,COMPLETED,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1077,False,NCT00866554,DUT112661,"[{'id': 'Health Canada-112661', 'type': 'OTHER...",CHU de Quebec-Universite Laval,OTHER,Efficacy and Toxicity of Bicalutamide and Duta...,Phase II Study of Bicalutamide and Dutasteride...,2023-05,COMPLETED,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1078,False,NCT04145323,2019-0557,"[{'id': 'SMPH/SURGERY/SURGTRAMA', 'type': 'OTH...","University of Wisconsin, Madison",OTHER,Novel Application of Indocyanine Green as a Bi...,Novel Application of Indocyanine Green as a Bi...,2020-01,COMPLETED,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
map_columns = {
    'protocolSection.identificationModule.nctId': 'NCT Number',
    'protocolSection.identificationModule.officialTitle': 'Study Title',
    'protocolSection.statusModule.startDateStruct.date': 'Start Date',
    'protocolSection.statusModule.studyFirstPostDateStruct.date': 'First Posted',
    'protocolSection.sponsorCollaboratorsModule.leadSponsor.name': 'Sponsor',
    'protocolSection.sponsorCollaboratorsModule.leadSponsor.class': 'Funder Type',
    'protocolSection.conditionsModule.conditions': 'Conditions',
    'protocolSection.designModule.studyType': 'Study Type',
    'protocolSection.designModule.phases': 'Phases',
    'protocolSection.designModule.enrollmentInfo.count': 'Enrollment',
    'protocolSection.armsInterventionsModule.interventions': 'Intervention/Intervention Type',
    'protocolSection.contactsLocationsModule.locations': 'Country',
    'protocolSection.sponsorCollaboratorsModule.collaborators': 'Collaborators',
    'protocolSection.statusModule.overallStatus': 'Study Status'
}

df_final = df_final.rename(columns=map_columns)

df_final_implantes = df_final[['NCT Number', 'Study Title', 'Start Date', 'First Posted', 'Sponsor', 'Funder Type', 'Conditions','Study Type','Phases','Enrollment','Intervention/Intervention Type','Country','Collaborators', 'Study Status']]

In [19]:
df_final_implantes

,NCT Number,Study Title,Start Date,First Posted,Sponsor,Funder Type,Conditions,Study Type,Phases,Enrollment,Intervention/Intervention Type,Country,Collaborators,Study Status
0,NCT01027637,In Situ Elasticity of Alloderm® in Breast Reco...,2010-01,2009-12-08,Henry Ford Health System,OTHER,[Breast Cancer],INTERVENTIONAL,[NA],33.0,"[{'type': 'OTHER', 'name': '5 mm vessel clips'...","[{'facility': 'Henry Ford Hospital', 'city': '...",NaN,COMPLETED
1,NCT02724371,A Study of the Safety and Effectiveness of the...,2016-04-01,2016-03-31,"Mentor Worldwide, LLC",INDUSTRY,"[Breast Reconstruction, Revision Breast Recons...",INTERVENTIONAL,[NA],400.0,"[{'type': 'DEVICE', 'name': 'Mentor Larger Siz...",[{'facility': 'Plastic Surgery Associates of M...,NaN,ACTIVE_NOT_RECRUITING
2,NCT03981263,Study of the Quality of Life in Patients With ...,2021-01-28,2019-06-10,"University Hospital, Toulouse",OTHER,[Breast Cancer],INTERVENTIONAL,[NA],64.0,"[{'type': 'DEVICE', 'name': 'MEAVANTI prothesi...","[{'facility': 'CHU Rangueil', 'city': 'Toulous...",NaN,COMPLETED
3,NCT01191164,A Prospective Study to Evaluate the Feasibilit...,2011-09,2010-08-30,British Columbia Cancer Agency,OTHER,[Breast Cancer],INTERVENTIONAL,[NA],5.0,"[{'type': 'PROCEDURE', 'name': 'Partial Breast...",[{'facility': 'British Columbia Cancer Agency'...,"[{'name': 'BC Cancer Foundation', 'class': 'OT...",UNKNOWN
4,NCT01793168,Coordination of Rare Diseases at Sanford,2010-07,2013-02-15,Sanford Health,OTHER,"[Rare Disorders, Undiagnosed Disorders, Disord...",OBSERVATIONAL,NaN,20000.0,NaN,"[{'facility': 'Sanford Health', 'status': 'REC...","[{'name': 'National Ataxia Foundation', 'class...",RECRUITING
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1074,NCT04608175,The Efficacy of Acupuncture as a Complementary...,2020-11,2020-10-29,University Ramon Llull,OTHER,"[Acupuncture Therapy, Breast Neoplasms]",INTERVENTIONAL,[NA],40.0,"[{'type': 'OTHER', 'name': 'Acupuncture', 'des...",NaN,"[{'name': 'Hospital Quiron Sagrado Corazon', '...",UNKNOWN
1076,NCT03734770,Patient's Preferences About Subcutaneous or Va...,2019-01-01,2018-11-08,Universita di Verona,OTHER,"[Progesterone, Luteal Phase Support, In Vitro ...",INTERVENTIONAL,[NA],149.0,"[{'type': 'DRUG', 'name': 'Subcutaneous Proges...",[{'facility': 'AOUI Verona - University of Ver...,NaN,COMPLETED
1077,NCT00866554,Phase II Study of Bicalutamide and Dutasteride...,2009-03,2009-03-20,CHU de Quebec-Universite Laval,OTHER,"[Prostate Cancer, Erectile Dysfunction, Lower ...",INTERVENTIONAL,[PHASE2],60.0,"[{'type': 'DRUG', 'name': 'administration of a...","[{'facility': 'CHUQ- Hotel-Dieu de Quebec', 'c...","[{'name': 'GlaxoSmithKline', 'class': 'INDUSTR...",COMPLETED
1078,NCT04145323,Novel Application of Indocyanine Green as a Bi...,2019-09-25,2019-10-30,"University of Wisconsin, Madison",OTHER,"[Breast Cancer, Necrosis]",INTERVENTIONAL,[EARLY_PHASE1],10.0,"[{'type': 'DIAGNOSTIC_TEST', 'name': 'ICG micr...","[{'facility': 'University of Wisconsin', 'city...",NaN,COMPLETED
